In [1]:
#Very first step is to import these modules and libraries into the satprocessing kernel
import numpy as np
import dask as ds
import earthaccess as ea
import xarray as xr
import sys
import matplotlib.pyplot as plt
import requests
import pandas as pd
import PyCO2SYS as pyco2
import cartopy
from shapely.ops import unary_union
import cartopy.feature as cfeature
from datetime import date
from scipy.spatial import KDTree
print("module import success")

import sys
sys.path.append(r'C:\\Users\\edmund.gable\\Documents\\Hollings_2026\\RESOURCES\\python\\utilities\\bootstrap\\')
print("path import success")

import pprint
from environment import bootstrap_environment  
pprint.pprint(bootstrap_environment(preferred="edmund_gable"))
print("success")

from utilities.bootstrap.environment import bootstrap_environment
env = bootstrap_environment(preferred="edmund.gable")
env
print("utilities import success")

#Before this next step, install the library 'resource': conda install -c conda-forge resource
from utilities import get_dates, get_source_file_dates, find_text
find_text("stats")

from utilities import get_prod_files 
get_prod_files("DO", period="DOY")
#"DOY" is climatological 'day of year
#'W' is climatological weekly files

from pathlib import Path

root = Path(r"C:\Users\edmund.gable\Documents\Hollings_2026\GitHub\MOM6-COBALT_2026")
raw_data = Path(r"C:\Users\edmund.gable\Documents\Hollings_2026\GitHub\MOM6-COBALT_2026\data\Raw_Data")
model_data = Path(r"Z:\\MOM6\Hindcasts\\BTM-O2-REGRID")




c:\Users\edmund.gable\.conda\envs\satprocessing\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


module import success
path import success
⚠ Unrecognized preferred label 'edmund_gable'. Valid options: ['NECMAC04363461.local', 'nefscsatdata', 'guihyde', 'Mac.localdomain', 'gdavis', 'egable']
{'dataset_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/DATASETS'),
 'hostname': 'NECL04363598',
 'lookuptable_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/workflow_resources/lookuptables'),
 'metadata_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/workflow_resources/metadata'),
 'project_root': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/python'),
 'python_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/python'),
 'satlogs_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/logs/satprocessing'),
 'timestamp': datetime.datetime(2026, 6, 11, 15, 17, 12, 374426),
 'utilities_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RE

In [2]:
#create the MOM6-COBALT grid df
df_mom6_raw=pd.read_csv(r"C:\Users\edmund.gable\Documents\Hollings_2026\GitHub\MOM6-COBALT_2026\data\Raw_Data\MOM6_COBALT\2004_01_01.csv")
grid_extent = df_mom6_raw[["lat", "lon"]].drop_duplicates().reset_index(drop=True)

# 3. Create a unique integer ID for every single unique grid cell
grid_extent["grid_id"] = grid_extent.index + 1  # 1-indexed IDs

# 4. Save this file so you don't have to keep rebuilding it
export_path=(r"C:\\Users\\edmund.gable\\Documents\\Hollings_2026\\GitHub\\MOM6-COBALT_2026\\data\\Index\\MOM6_Grid_Extent.csv")
grid_extent.to_csv(export_path, date_format='%Y-%m-%d %H:%M:%S', index=False)


print(f"Grid extent file created with {len(grid_extent)} unique grid IDs.")
print(grid_extent.head())

PermissionError: [Errno 13] Permission denied: 'C:\\\\Users\\\\edmund.gable\\\\Documents\\\\Hollings_2026\\\\GitHub\\\\MOM6-COBALT_2026\\\\data\\\\Index\\\\MOM6_Grid_Extent.csv'

In [32]:
def classify_DO(df, DO_column, battery_col=None):
  clean_df=df[df[DO_column]>0].copy()
  classified_df = clean_df.copy()

  classified_df.loc[classified_df[DO_column] >= 3.9, 'DO_status'] = 'Low Oxygen'
  classified_df.loc[classified_df[DO_column] > 5.0, 'DO_status'] = 'Oxygenated' 
  classified_df.loc[classified_df[DO_column] < 3.9, 'DO_status'] = 'Mildly Hypoxic'
  classified_df.loc[classified_df[DO_column] < 2.41, 'DO_status'] = 'Hypoxia'
  classified_df.loc[classified_df[DO_column] < 0.2, 'DO_status'] = 'Anoxia'

    
  # 2. FILTER BATTERY (Only if a battery column name was provided)
  if battery_col is not None:
    classified_df = classified_df[classified_df[battery_col] >= 50]
        
    classified_df = classified_df.reset_index(drop=True)
    
  return classified_df
  
def open_file(file):
  ds=xr.open_dataset(file)
  df=ds.to_dataframe()
  print(df.columns.to_list())
  return df

def check_units(ds, DO_col):
  print(ds[DO_col].attrs)


def prepare_df(df, DO_col, id_col, obs_depth_col, bottom_depth_col, lat_col, lon_col, time_col, mom6_grid_df, battery_col=None):
  #drop values that are nan
  df=df.dropna(subset=[DO_col])
  #classify the oxygen level
  df_working=classify_DO(df, DO_col, battery_col=None)
  #make sure measurement is at bottom
  depth_mismatch = abs(df_working[obs_depth_col] - df_working[bottom_depth_col])
  df_filtered=df_working[depth_mismatch<10].copy()
  #match to MOM6 grid timestamp
  df_filtered["matched_time"] = pd.to_datetime(df_filtered[time_col]).dt.normalize() + pd.Timedelta(hours=12)
  #match to spatial grids
  model_coords=mom6_grid_df[['lat','lon']].drop_duplicates().values
  tree=KDTree(model_coords)
  raw_coords=df_filtered[[lat_col, lon_col]].values
  _, closest_indices=tree.query(raw_coords)
  matched_lat_lon=model_coords[closest_indices]

  df_filtered['matched_lat']=matched_lat_lon[:, 0]
  df_filtered['matched_lon']=matched_lat_lon[:, 1]
  return df_filtered

import pandas as pd
from scipy.spatial import KDTree


def prepare_df(df, DO_col, id_col, obs_depth_col, bottom_depth_col, lat_col, lon_col, time_col, mom6_grid_df, battery_col=None):
    # 1. Drop values that are NaN in the target DO column
    df = df.dropna(subset=[DO_col])

    # 2. Classify the oxygen level (Note: changed battery_col=None to battery_col=battery_col to pass it through)
    df_working = classify_DO(df, DO_col, battery_col=battery_col)

    # 3. Make sure measurement is near the bottom (< 10 meters)
    depth_mismatch = abs(df_working[obs_depth_col] - df_working[bottom_depth_col])
    df_filtered = df_working[depth_mismatch < 10].copy()

    # 4. Match to MOM6 grid timestamp (Daily normalized to 12:00:00)
    df_filtered["matched_time"] = pd.to_datetime(df_filtered[time_col]).dt.normalize() + pd.Timedelta(hours=12)

    # 5. Match to spatial grids using KD-Tree
    # We keep the grid_id tied to the lat/lon pairs in the model array
    model_spatial = (mom6_grid_df[["lat", "lon", "grid_id"]].drop_duplicates().copy())
    model_coords = model_spatial[["lat", "lon"]].values

    tree = KDTree(model_coords)
    raw_coords = df_filtered[[lat_col, lon_col]].values
    _, closest_indices = tree.query(raw_coords)

    # 6. Extract matched coordinates and grid IDs
    matched_lat_lon = model_coords[closest_indices]
    df_filtered["matched_lat"] = matched_lat_lon[:, 0]
    df_filtered["matched_lon"] = matched_lat_lon[:, 1]

    # Map the unique grid_id from the corresponding tree positions
    df_filtered["grid_id"] = (model_spatial["grid_id"].iloc[closest_indices].values)

    # 7. Rename the raw columns to your requested standardized names
    rename_dict = {
        DO_col: "dissolved_oxygen",
        id_col: "id",
        obs_depth_col: "obs_depth",
        bottom_depth_col: "bottom_depth",
        lat_col: "lat",
        lon_col: "lon",
        time_col: "time",
    }
    df_filtered = df_filtered.rename(columns=rename_dict)

    # convert the MOM6-COBALT data to mg/L from mol/kg
    dfm["btm_o2"] = dfm["btm_o2"] * 31998 * 1.025

    # 8. Define the final strict list of columns to return
    requested_columns = [
        "id",
        "time",
        "lat",
        "lon",
        "obs_depth",
        "bottom_depth",
        "dissolved_oxygen",
        "DO_status",  # Included this so you don't lose the tags from classify_DO!
        "grid_id",
        "matched_time",
        "matched_lat",
        "matched_lon",
        btm_o2
    ]

    # Return the dataframe filtered tightly down to ONLY these columns
    return df_filtered[requested_columns]

  



    

In [33]:
file=(r"C:\Users\edmund.gable\Documents\Hollings_2026\GitHub\MOM6-COBALT_2026\data\Raw_Data\Hydrographic_Monitoring\errdap_search.nc")
df=open_file(file)

check_units(df, 'dissolved_oxygen')
df=df[df['cruise_id']!='HB2101']
classified_df=classify_DO(df, 'dissolved_oxygen')
mom6_grid_df=pd.read_csv(r"C:\Users\edmund.gable\Documents\Hollings_2026\GitHub\MOM6-COBALT_2026\data\Index\MOM6_Grid_Extent.csv")

dfp=prepare_df(df, 'dissolved_oxygen', 'cruise_id', 'depth', 'bottom_depth', 'latitude', 'longitude', 'UTC_DATETIME', mom6_grid_df)
dfp

['UTC_DATETIME', 'latitude', 'longitude', 'depth', 'dissolved_oxygen', 'cruise_id', 'purpose_code', 'bottom_depth', 'GEAR_TYPE']
{}


NameError: name 'dfm' is not defined

In [34]:
ds=xr.open_dataset(r"C:\Users\edmund.gable\Documents\Hollings_2026\GitHub\MOM6-COBALT_2026\data\Raw_Data\Hydrographic_Monitoring\errdap_search.nc")
nearest_grid = ds.sel(lat=pt_lats, lon=pt_lons, method='nearest')

NameError: name 'pt_lats' is not defined